In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
import os
os.chdir("/content")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
# !rm -rf "/content/vlm-safety-reasoning"

In [5]:
import subprocess
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/vlm-finetuning-project1"
REPO_DIR = "vlm-safety-reasoning"
ENV_PATH = f"{DRIVE_ROOT}/secrets/.env"

def load_secrets(env_path: str) -> dict:
    """Read a .env file and export its values into os.environ."""
    if not os.path.exists(env_path):
        raise FileNotFoundError(f"Secrets file not found at: {env_path}")

    secrets = {}
    with open(env_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            secrets[key] = value.strip(" \"'\r")
            os.environ[key] = secrets[key]
    return secrets


print(">>> Loading secrets...")
secrets = load_secrets(ENV_PATH)
required_keys = ["GIT_EMAIL", "GIT_NAME", "GITHUB_USERNAME", "GITHUB_TOKEN", "HF_TOKEN"]
missing = [k for k in required_keys if k not in secrets]
if missing:
    raise KeyError(f"Missing required secrets: {missing}")
print(">>> Secrets loaded successfully.")

print(">>> Configuring Git identity...")
subprocess.run(["git", "config", "--global", "user.email", secrets["GIT_EMAIL"]], check=True)
subprocess.run(["git", "config", "--global", "user.name", secrets["GIT_NAME"]], check=True)

AUTH_REPO_URL = (
    f"https://{secrets['GITHUB_USERNAME']}:{secrets['GITHUB_TOKEN']}"
    f"@github.com/epmresearch/vlm-safety-reasoning.git"
)

if os.path.exists(REPO_DIR):
    print(">>> Repo already present, pulling latest...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "remote", "set-url", "origin", AUTH_REPO_URL], check=True)
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    print(">>> Cloning repo...")
    subprocess.run(["git", "clone", AUTH_REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print(f">>> Working directory: {os.getcwd()}")

print(">>> Copying .env into local workspace...")
shutil.copy(ENV_PATH, ".env")

print(">>> Installing requirements...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(">>> Setup complete.")

>>> Loading secrets...
>>> Secrets loaded successfully.
>>> Configuring Git identity...
>>> Cloning repo...
>>> Working directory: /content/vlm-safety-reasoning
>>> Copying .env into local workspace...
>>> Installing requirements...
>>> Setup complete.


In [18]:
import subprocess, shutil as sh

print("=== Ensuring Java 8 is default (required by SPICE's bundled CoreNLP 3.6.0) ===")
subprocess.run(["apt-get", "install", "-y", "-q", "openjdk-8-jdk-headless"], check=True)

listing = subprocess.run(["update-alternatives", "--list", "java"], capture_output=True, text=True).stdout
print(listing)

java8_candidates = [line.strip() for line in listing.splitlines() if "java-8" in line]
assert java8_candidates, f"No java-8 path found in update-alternatives output:\n{listing}"
JAVA8_PATH = java8_candidates[0]
print(f"Using: {JAVA8_PATH}")

subprocess.run(["update-alternatives", "--set", "java", JAVA8_PATH], check=True)

verify = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
print(verify)
assert "1.8" in verify, f"Java switch failed, still got: {verify}"
print("  Java 8 is now the active `java`")

=== Ensuring Java 8 is default (required by SPICE's bundled CoreNLP 3.6.0) ===
/usr/lib/jvm/java-17-openjdk-amd64/bin/java
/usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java

Using: /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java
openjdk version "1.8.0_492"
OpenJDK Runtime Environment (build 1.8.0_492-8u492-ga~us2-0ubuntu1~22.04.1-b09)
OpenJDK 64-Bit Server VM (build 25.492-b09, mixed mode)

  Java 8 is now the active `java`


In [19]:
java_path = sh.which("java")
assert java_path is not None, "Java missing from PATH — METEOR/CIDEr-D/SPICE will silently return {}"

from evaluation.metrics_captioning import _check_java_available
assert _check_java_available(), "Pipeline's internal Java check failed"
print("  Java available:", java_path)

  Java available: /usr/bin/java


In [20]:
import importlib

required_pkgs = ["pycocoevalcap", "bert_score", "transformers", "torch", "sentence_transformers"]
for pkg in required_pkgs:
    try:
        importlib.import_module(pkg)
        print(f"[OK] {pkg}")
    except ImportError as e:
        print(f"[MISSING] {pkg}: {e}")

[OK] pycocoevalcap
[OK] bert_score
[OK] transformers
[OK] torch
[OK] sentence_transformers


In [21]:
from evaluation.spice_cache import restore_spice_cache, save_spice_cache
from core.io import get_drive_path

SPICE_CACHE_DIR = str(get_drive_path("tools", "spice_corenlp_cache"))
restored = restore_spice_cache(SPICE_CACHE_DIR)
print(f"SPICE cache restored: {restored}")

2026-07-21 10:43:43 | INFO     | evaluation.spice_cache:restore_spice_cache:54 - No SPICE cache found at /content/drive/MyDrive/vlm-finetuning-project1/tools/spice_corenlp_cache. First run will download CoreNLP models (~2GB, one-time).
SPICE cache restored: False


In [22]:
from evaluation.output_parser import strip_fences, parse_model_output, validate_unified_output

test_cases = [
    ('```json\n{"caption": "test"}\n```', "standard fence"),
    ('```\n{"a": 1}\n```', "no json identifier"),
    ('Here you go:\n```json\n{"caption": "safe"}\n```\nHope this helps!', "preamble+postamble"),
    ('  {"a": 1}  ', "no fences at all"),
    ('```json\n{"caption": "A construction site with', "TRUNCATED"),
    ('', "empty string"),
    ('I cannot see any violations.', "pure hallucination"),
]
for raw, label in test_cases:
    parsed = parse_model_output(raw)
    valid = validate_unified_output(parsed) is not None if parsed else False
    print(f"{label:<40} | parses={parsed is not None} | schema_valid={valid}")

assert parse_model_output('```json\n{"caption": "A construction site with```') is None
assert validate_unified_output({"excavator": []}) is None
print("\n  Output parser edge cases OK")

standard fence                           | parses=True | schema_valid=True
2026-07-21 10:44:05 | WARNING  | evaluation.output_parser:validate_unified_output:51 - Failed to validate UnifiedOutput schema: 1 validation error for UnifiedOutput
caption
  Field required [type=missing, input_value={'a': 1}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
no json identifier                       | parses=True | schema_valid=False
preamble+postamble                       | parses=True | schema_valid=True
2026-07-21 10:44:05 | WARNING  | evaluation.output_parser:validate_unified_output:51 - Failed to validate UnifiedOutput schema: 1 validation error for UnifiedOutput
caption
  Field required [type=missing, input_value={'a': 1}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
no fences at all                         | parses=True | schema_valid=False
TRUNCATED                                | parses=False 

In [23]:
from data.box_utils import (
    normalize_boxes, clean_boxes, compute_iou, greedy_multibox_iou,
    scale_01_to_1000, scale_1000_to_01,
)

checks = [
    ("flat box -> nested", normalize_boxes([10, 20, 30, 40]) == [[10, 20, 30, 40]]),
    ("dict wrapper", normalize_boxes([{"bounding_box": [10, 20, 30, 40]}]) == [[10, 20, 30, 40]]),
    ("zero-area filtered", clean_boxes([[10, 20, 10, 40]]) == []),
]
iou, _, _ = compute_iou([0, 0, 1, 1], [0, 0, 1, 1])
checks.append(("perfect IoU = 1.0", abs(iou - 1.0) < 1e-9))
iou, _, _ = compute_iou([0, 0, 1, 0.5], [0, 0, 1, 1])
checks.append(("known partial overlap = 0.5", abs(iou - 0.5) < 1e-9))
rt = scale_1000_to_01(scale_01_to_1000([0.1, 0.2, 0.9, 0.8]))
checks.append(("scale roundtrip", all(abs(a - b) < 1e-3 for a, b in zip(rt, [0.1, 0.2, 0.9, 0.8]))))
iou, _, _ = greedy_multibox_iou([], [])
checks.append(("TN IoU == 0.0", iou == 0.0))
iou, _, _ = greedy_multibox_iou([], [[0, 0, 1, 1]])
checks.append(("FN IoU == 0.0", iou == 0.0))

all_pass = True
for label, result in checks:
    print(f"{label:<40} | {'PASS' if result else 'FAIL'}")
    all_pass &= result
assert all_pass
print("\n  box_utils verified")

flat box -> nested                       | PASS
dict wrapper                             | PASS
zero-area filtered                       | PASS
perfect IoU = 1.0                        | PASS
known partial overlap = 0.5              | PASS
scale roundtrip                          | PASS
TN IoU == 0.0                            | PASS
FN IoU == 0.0                            | PASS

  box_utils verified


In [24]:
from evaluation.metrics_structural import compute_structural_metrics

outputs = [
    '```json\n{"caption": "safe site"}\n```',
    'This is conversational hallucination.',
    '{"random_key": true}',
]
res = compute_structural_metrics(outputs)
print(res)
assert abs(res["structural_json_validity_rate"] - 2/3) < 1e-9
assert abs(res["structural_schema_adherence_rate"] - 1/3) < 1e-9
print("  structural metrics OK")

2026-07-21 10:44:32 | WARNING  | evaluation.output_parser:validate_unified_output:51 - Failed to validate UnifiedOutput schema: 1 validation error for UnifiedOutput
caption
  Field required [type=missing, input_value={'random_key': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
{'structural_json_validity_rate': 0.6666666666666666, 'structural_schema_adherence_rate': 0.3333333333333333, 'structural_valid_json_count': 2, 'structural_valid_schema_count': 1, 'structural_total_samples_count': 3}
  structural metrics OK


In [25]:
from evaluation.metrics_captioning import compute_bertscore, compute_meteor, compute_cider, compute_spice

preds = ["A yellow excavator is digging in a construction site with two workers nearby."]
refs  = ["An excavator digs at a construction site while two workers stand nearby."]

print("=== BERTScore ===")
bert_res = compute_bertscore(preds, refs)
print(bert_res)
assert bert_res and 0.0 <= bert_res["bertscore_f1"] <= 1.0

print("\n=== METEOR ===")
meteor_res = compute_meteor(preds, refs)
print(meteor_res)
assert meteor_res

print("\n=== CIDEr-D (single pair — near-zero is EXPECTED, not a bug, see multi-pair check below) ===")
cider_res = compute_cider(preds, refs)
print(cider_res)

print("\n=== CIDEr-D (multi-pair — this is the real check) ===")
preds_multi = [
    "A yellow excavator is digging in a construction site with two workers nearby.",
    "Two workers wearing hard hats are standing near a pile of rebar.",
    "A crane is lifting steel beams over a partially built structure.",
]
refs_multi = [
    "An excavator digs at a construction site while two workers stand nearby.",
    "Workers in hard hats stand next to stacked rebar.",
    "A crane lifts steel beams above an unfinished building.",
]
cider_multi = compute_cider(preds_multi, refs_multi)
print(cider_multi)
assert cider_multi["ciderd"] > 0.0, "CIDEr-D still zero on a multi-sample batch — real bug"

print("\n=== SPICE ===")
spice_res = compute_spice(preds, refs)
print(spice_res)
assert spice_res, "SPICE returned empty"
print("  all captioning sub-metrics working")

=== BERTScore ===


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

{'bertscore_precision': 0.7866341471672058, 'bertscore_recall': 0.8257765769958496, 'bertscore_f1': 0.8064541816711426}

=== METEOR ===
{'meteor': 0.31928273154609327}

=== CIDEr-D (single pair — near-zero is EXPECTED, not a bug, see multi-pair check below) ===
{'ciderd': 0.0}

=== CIDEr-D (multi-pair — this is the real check) ===
{'ciderd': 1.5647668316096972}

=== SPICE ===
{'spice': 0.6666666666666666, 'spice_object_f1': 1.0, 'spice_relation_f1': 0.0, 'spice_attribute_f1': 0.6666666666666666, 'spice_cardinality_f1': 1.0}
  all captioning sub-metrics working


In [26]:
save_spice_cache(SPICE_CACHE_DIR)
print("  SPICE cache saved — future sessions won't redownload the 384MB CoreNLP models")

2026-07-21 10:45:20 | INFO     | evaluation.spice_cache:save_spice_cache:105 - Saved SPICE cache to Drive: 19 new item(s) -> /content/drive/MyDrive/vlm-finetuning-project1/tools/spice_corenlp_cache
  SPICE cache saved — future sessions won't redownload the 384MB CoreNLP models


In [27]:
from evaluation.metrics_captioning import compute_clipscore
from data.loader import load_processed_dataset

splits = load_processed_dataset()

2026-07-21 10:45:50 | INFO     | data.loader:load_processed_dataset:183 - Loading fully processed dataset from disk: /content/drive/MyDrive/vlm-finetuning-project1/datasets/processed
2026-07-21 10:46:19 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'train' split: 6308 samples
2026-07-21 10:46:19 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'val' split: 701 samples
2026-07-21 10:46:19 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'test' split: 3004 samples


In [28]:
real_img = splits["test"][0]["image"]
real_caption = splits["test"][0]["image_caption"]

clip_res = compute_clipscore([real_caption], [real_img])
print(clip_res)
assert clip_res

wrong_caption = "A photo of a cat sleeping on a sofa."
clip_wrong = compute_clipscore([wrong_caption], [real_img])
print(f"True: {clip_res['clipscore']:.3f}  Wrong: {clip_wrong['clipscore']:.3f}")
assert clip_res["clipscore"] > clip_wrong["clipscore"]
print("  CLIPScore correctly ranks true > wrong caption")

2026-07-21 10:46:19 | INFO     | evaluation.metrics_captioning:compute_clipscore:219 - Computing CLIPScore (batched)...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

{'clipscore': 0.8123967796564102}
2026-07-21 10:46:29 | INFO     | evaluation.metrics_captioning:compute_clipscore:219 - Computing CLIPScore (batched)...
True: 0.812  Wrong: 0.408
  CLIPScore correctly ranks true > wrong caption


In [ ]:
preds_multi = [
    "A yellow excavator is digging in a construction site with two workers nearby.",
    "Two workers wearing hard hats are standing near a pile of rebar.",
    "A crane is lifting steel beams over a partially built structure.",
]
refs_multi = [
    "An excavator digs at a construction site while two workers stand nearby.",
    "Workers in hard hats stand next to stacked rebar.",
    "A crane lifts steel beams above an unfinished building.",
]
cider_multi = compute_cider(preds_multi, refs_multi)
print(cider_multi)
assert cider_multi["ciderd"] > 0.0, "Still zero with a multi-sample batch — now THIS would be a real bug"

In [8]:
from evaluation.metrics_captioning import compute_all_caption_metrics

agg = compute_all_caption_metrics(preds, refs, images=[real_img], prefix="captioning_")
print(agg)

expected = ["captioning_bertscore_f1", "captioning_meteor", "captioning_ciderd",
            "captioning_spice", "captioning_clipscore"]
missing = [k for k in expected if k not in agg]
assert not missing, f"Aggregator missing: {missing}"
print("  captioning aggregator OK")

Case                                          | strip_fences OK  | parses   | schema valid
-----------------------------------------------------------------------------------------------
standard fence                                | True             | True     | True
2026-07-21 10:29:09 | WARNING  | evaluation.output_parser:validate_unified_output:51 - Failed to validate UnifiedOutput schema: 1 validation error for UnifiedOutput
caption
  Field required [type=missing, input_value={'a': 1}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
no json identifier                            | True             | True     | False
preamble+postamble                            | True             | True     | True
2026-07-21 10:29:09 | WARNING  | evaluation.output_parser:validate_unified_output:51 - Failed to validate UnifiedOutput schema: 1 validation error for UnifiedOutput
caption
  Field required [type=missing, input_value={'a': 1}, input_type=dict

In [9]:
from evaluation.metrics_grounding import compute_grounding_metrics

# Image 1: excavator perfectly predicted. Image 2: excavator FP (hallucinated), rebar poorly predicted.
refs_g = [
    {"excavator": [[0, 0, 0.5, 0.5]]},
    {"rebar": [[0.5, 0.5, 1.0, 1.0]]},
]
preds_g = [
    {"excavator": [[0, 0, 500, 500]]},                          # perfect (1000-scale)
    {"rebar": [[0, 0, 100, 100]], "excavator": [[0, 0, 1000, 1000]]},  # poor rebar, FP excavator
]

res = compute_grounding_metrics(preds_g, refs_g)
# Hand math: excavator total macro = (1.0 + 0.0)/2 = 0.5 ; existing macro (img1 only) = 1.0
print("excavator all_macro (expect 0.5):     ", res["grounding_iou_all_macro_excavator_tn0"])
print("excavator existing_macro (expect 1.0):", res["grounding_iou_existing_macro_excavator_tn0"])
assert res["grounding_iou_all_macro_excavator_tn0"] == 0.5
assert res["grounding_iou_existing_macro_excavator_tn0"] == 1.0
print("  grounding metrics match hand-computed values")

Check                                         | Result
------------------------------------------------------------
flat box -> nested                            | PASS
dict wrapper                                  | PASS
zero-area filtered                            | PASS
perfect IoU = 1.0                             | PASS
known partial overlap = 0.5                   | PASS
scale roundtrip                               | PASS
TN (empty/empty) IoU == 0.0 per N3 fix        | PASS
FN (gt has box, pred empty) IoU == 0.0        | PASS

  box_utils math verified


In [10]:
from evaluation.metrics_structural import compute_structural_metrics

outputs = [
    '```json\n{"caption": "safe site"}\n```',                 # valid JSON + valid schema
    'This is conversational hallucination.',                    # invalid JSON
    '{"random_key": true}',                                     # valid JSON, missing caption -> invalid schema
]
res = compute_structural_metrics(outputs)
print(res)
assert abs(res["structural_json_validity_rate"] - 2/3) < 1e-9
assert abs(res["structural_schema_adherence_rate"] - 1/3) < 1e-9
print("  structural metrics match hand-computed expectation (2/3 valid JSON, 1/3 valid schema)")

2026-07-21 10:30:18 | WARNING  | evaluation.output_parser:validate_unified_output:51 - Failed to validate UnifiedOutput schema: 1 validation error for UnifiedOutput
caption
  Field required [type=missing, input_value={'random_key': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
{'structural_json_validity_rate': 0.6666666666666666, 'structural_schema_adherence_rate': 0.3333333333333333, 'structural_valid_json_count': 2, 'structural_valid_schema_count': 1, 'structural_total_samples_count': 3}
  structural metrics match hand-computed expectation (2/3 valid JSON, 1/3 valid schema)


In [12]:
from evaluation.metrics_captioning import (
    compute_bertscore, compute_meteor, compute_cider, compute_spice, compute_clipscore
)

preds_multi = [
    "A yellow excavator is digging in a construction site with two workers nearby.",
    "Two workers wearing hard hats are standing near a pile of rebar.",
    "A crane is lifting steel beams over a partially built structure.",
]
refs_multi = [
    "An excavator digs at a construction site while two workers stand nearby.",
    "Workers in hard hats stand next to stacked rebar.",
    "A crane lifts steel beams above an unfinished building.",
]
cider_multi = compute_cider(preds_multi, refs_multi)
print(cider_multi)
assert cider_multi["ciderd"] > 0.0, "Still zero with a multi-sample batch — now THIS would be a real bug"

{'ciderd': 1.5647668316096972}


In [11]:
from evaluation.metrics_captioning import (
    compute_bertscore, compute_meteor, compute_cider, compute_spice, compute_clipscore
)

preds = ["A yellow excavator is digging in a construction site with two workers nearby."]
refs  = ["An excavator digs at a construction site while two workers stand nearby."]

print("=== BERTScore (downloads RoBERTa-large on first call — watch for OOM/network errors) ===")
bert_res = compute_bertscore(preds, refs)
print(bert_res)
assert bert_res, "BERTScore returned empty dict — check bert_score/transformers install"
assert 0.0 <= bert_res["bertscore_f1"] <= 1.0

print("\n=== METEOR (Java-backed, official jar) ===")
meteor_res = compute_meteor(preds, refs)
print(meteor_res)
assert meteor_res, "METEOR returned empty — Java check passed above but this still failed, investigate"

print("\n=== CIDEr-D (Java-backed PTBTokenizer) ===")
cider_res = compute_cider(preds, refs)
print(cider_res)
assert cider_res, "CIDEr-D returned empty"

print("\n=== SPICE (Java + CoreNLP models, slow first call) ===")
spice_res = compute_spice(preds, refs)
print(spice_res)
assert spice_res, "SPICE returned empty — check CoreNLP download completed"
# sub-category scores should also be present
print("SPICE sub-categories present:", [k for k in spice_res if k.startswith("spice_")])

=== BERTScore (downloads RoBERTa-large on first call — watch for OOM/network errors) ===


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

{'bertscore_precision': 0.7866341471672058, 'bertscore_recall': 0.8257765769958496, 'bertscore_f1': 0.8064541816711426}

=== METEOR (Java-backed, official jar) ===
{'meteor': 0.31928273154609327}

=== CIDEr-D (Java-backed PTBTokenizer) ===
{'ciderd': 0.0}

=== SPICE (Java + CoreNLP models, slow first call) ===
Progress: 384.5M / 384.5M (100.0%)
Extracting stanford-corenlp-3.6.0 ...
Done.
2026-07-21 10:32:28 | WARNING  | evaluation.metrics_captioning:compute_spice:174 - Failed to compute SPICE: Command '['java', '-jar', '-Xmx8G', 'spice-1.0.jar', '/usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/tmp/tmprdvl8dot', '-cache', '/usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/cache', '-out', '/usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/tmp/tmp8uhq5kfe', '-subset', '-silent']' returned non-zero exit status 1.
{}


AssertionError: SPICE returned empty — check CoreNLP download completed

In [13]:
import subprocess

java_path = subprocess.run(["which", "java"], capture_output=True, text=True).stdout.strip()
print("java path:", java_path)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

java path: /usr/bin/java
openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)



In [15]:
import subprocess

java_path = subprocess.run(["which", "java"], capture_output=True, text=True).stdout.strip()
print("java path:", java_path)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

import pycocoevalcap.spice.spice as spice_module
from pathlib import Path
spice_dir = Path(spice_module.__file__).resolve().parent
print("SPICE dir:", spice_dir)

# Re-run the same command your traceback showed, but capture output this time
result = subprocess.run(
    ["java", "-jar", "-Xmx8G", "spice-1.0.jar",
     # use whatever tmp input file path appeared in your error, or just check the jar loads at all:
     "-h" if False else "spice-1.0.jar"],  # placeholder — see step 3 below for the real invocation
    cwd=str(spice_dir),
    capture_output=True, text=True,
)
print("STDOUT:", result.stdout[:2000])
print("STDERR:", result.stderr[:2000])

print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

java path: /usr/bin/java
openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)

SPICE dir: /usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice
STDOUT: ********  SPICE Evaluation  ********

All tuples
  f-score:	NaN (SPICE metric)
  precision:	NaN
  recall:	NaN
  true pos:	0
  false pos:	0
  false neg:	0
  num images:	0

SPICE evaluation took: 522.4 ms

STDERR: Could not read input: spice-1.0.jar
Unexpected character (P) at position 0.
Unexpected character (P) at position 0.
	at org.json.simple.parser.Yylex.yylex(Yylex.java:610)
	at org.json.simple.parser.JSONParser.nextToken(JSONParser.java:269)
	at org.json.simple.parser.JSONParser.parse(JSONParser.java:118)
	at org.json.simple.parser.JSONParser.parse(JSONParser.java:92)
	at edu.anu.spice.SpiceScorer.scoreBatch(SpiceScorer.java:91)
	at edu.anu.spice.SpiceScorer.main(SpiceScorer.java:60)
Parsing refer

In [16]:
import subprocess
subprocess.run(["apt-get", "install", "-y", "-q", "openjdk-8-jdk-headless"], check=True)

result = subprocess.run(["update-alternatives", "--list", "java"], capture_output=True, text=True)
print(result.stdout)

JAVA8_PATH = "/usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java"  # copy the exact path from Step 1's output

subprocess.run(["update-alternatives", "--set", "java", JAVA8_PATH], check=True)

verify = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(verify.stderr)
assert "1.8" in verify.stderr, f"Still not Java 8, got: {verify.stderr}"

/usr/lib/jvm/java-17-openjdk-amd64/bin/java
/usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java

openjdk version "1.8.0_492"
OpenJDK Runtime Environment (build 1.8.0_492-8u492-ga~us2-0ubuntu1~22.04.1-b09)
OpenJDK 64-Bit Server VM (build 25.492-b09, mixed mode)



In [17]:
from evaluation.metrics_captioning import compute_spice

preds = ["A yellow excavator is digging in a construction site with two workers nearby."]
refs  = ["An excavator digs at a construction site while two workers stand nearby."]

spice_res = compute_spice(preds, refs)
print(spice_res)
assert spice_res, "SPICE still empty after switching to Java 8"
print("✅ SPICE working:", spice_res)

{'spice': 0.6666666666666666, 'spice_object_f1': 1.0, 'spice_relation_f1': 0.0, 'spice_attribute_f1': 0.6666666666666666, 'spice_cardinality_f1': 1.0}
✅ SPICE working: {'spice': 0.6666666666666666, 'spice_object_f1': 1.0, 'spice_relation_f1': 0.0, 'spice_attribute_f1': 0.6666666666666666, 'spice_cardinality_f1': 1.0}


In [ ]:
import json
from data.loader import load_processed_dataset
from data.preprocessor import build_ground_truth_dict
from evaluation.evaluator import run_full_evaluation
from core.io import get_drive_path, ensure_dir

MODEL_TIER = "2b"
DRIVE_RESULTS_DIR = get_drive_path("results", f"baseline_test_full_{MODEL_TIER}_second")
ensure_dir(DRIVE_RESULTS_DIR)
JSONL_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "predictions.jsonl")

In [ ]:
# Load the raw inference results
results = []
with open(JSONL_OUTPUT_PATH, "r") as f:
    for line in f:
        results.append(json.loads(line))

results[0]

In [ ]:
# Extract predictions and references
raw_predictions = [res["raw_output"] for res in results]
references = [build_ground_truth_dict(res["sample"]) for res in results]

In [ ]:
# Load the HuggingFace dataset to get the physical PIL images back
splits = load_processed_dataset()
test_data = splits["test"]

In [ ]:
# Safely map the images using image_id so they align perfectly
image_map = {str(sample["image_id"]): sample["image"] for sample in test_data}
images = [image_map.get(str(res.get("image_id"))) for res in results]

In [ ]:
# Run the evaluation
print("Running evaluation pipeline...")
eval_results = run_full_evaluation(raw_predictions, references, images=images)

# Look at your final scores!
print(eval_results["metrics"])

If you decide to split it into two separate notebooks (which is actually a very smart idea to prevent losing data if Colab crashes during evaluation), you just have to do the exact same image_id mapping when you load the data back in.

Here is the exact code snippet you should use in your standalone Evaluation Notebook to safely load your saved JSONL file and run the evaluator without any misalignment:

By explicitly building that image_map dictionary and pulling the images out using res.get("image_id"), you guarantee that your text and images are locked together perfectly, no matter what order the JSONL file was saved in!

In [ ]:
# ============================================================
# Java + SPICE/CIDEr-D/METEOR setup — ONLY needed in evaluation
# notebooks. Do NOT add this to training or plain-inference notebooks.
# ============================================================
print(">>> Installing JRE (needed for official CIDEr-D/METEOR/SPICE)...")
!apt-get -qq update && apt-get -qq install -y default-jre
!java -version

from evaluation.spice_cache import restore_spice_cache
from core.io import get_drive_path

SPICE_CACHE_DIR = str(get_drive_path("tools", "spice_corenlp_cache"))
cache_restored = restore_spice_cache(SPICE_CACHE_DIR)
print(f">>> SPICE/CoreNLP cache restored from Drive: {cache_restored}")
if not cache_restored:
    print(">>> First-ever run: SPICE will download ~2GB of CoreNLP models "
          "on its first call. This is one-time only — remember to run "
          "save_spice_cache() afterward (see cell below).")

In [ ]:
# test
from evaluation.metrics_captioning import compute_all_caption_metrics

test_preds = ["A yellow excavator is digging near a pile of dirt."]
test_refs = ["An excavator digs beside a mound of soil."]

test_result = compute_all_caption_metrics(test_preds, test_refs, include_spice=True)
print(test_result)
assert "cider" in test_result and "meteor" in test_result and "spice" in test_result, \
    "One or more official metrics failed silently — check the warnings above."
print(">>> Java toolchain OK.")

After your first full evaluation run completes (so the CoreNLP jars have definitely been downloaded), run this once:
pythonfrom evaluation.spice_cache import save_spice_cache
save_spice_cache(SPICE_CACHE_DIR)
From then on, every future session's restore_spice_cache() call at the top will find the cache and skip the 2GB download.

In [ ]:
from evaluation.spice_cache import save_spice_cache
save_spice_cache(SPICE_CACHE_DIR)

In [ ]:
import os
import json
from evaluation.evaluator import run_full_evaluation

METRICS_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "metrics.json")
FAILURES_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "json_schema_failures.jsonl")
PARSED_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "parsed_predictions.jsonl")

In [ ]:
with open(METRICS_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(final_eval_payload["metrics"], f, indent=2, ensure_ascii=False)
with open(FAILURES_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for failure in final_eval_payload.get("failures", []):
        f.write(json.dumps(failure) + "\n")
with open(PARSED_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for pred in final_eval_payload.get("parsed_predictions", []):
        if pred is not None:
            f.write(json.dumps(pred) + "\n")
print(f"All artifacts perfectly saved to: {DRIVE_RESULTS_DIR}")